# Lab 19 — Online evaluation and tail-based sampling

Two halves, one lab. Half A wires the LangSmith Python SDK polling pattern. Half B walks through a real OTel Collector `tail_sampling` config and simulates the policy logic in Python.

Self-contained: synthetic traces via `client.create_run` (no need to re-run Labs 17/18). The patterns apply identically to real production traffic.

> 📖 Required reading: [`concepts/evaluation/online-evaluator-registration.md`](../../concepts/evaluation/online-evaluator-registration.md), [`concepts/evaluation/tail-based-sampling.md`](../../concepts/evaluation/tail-based-sampling.md).
> ⬅️ Recommended: at least one of [Lab 17](../17-langsmith-trace-ingestion/) or [Lab 18](../18-opentelemetry-portable-tracing/).
> 🛠 LangSmith free-tier account; this lab uses ~10 traces.
> ⏱ Run time: 80-100 min including reading.


## Step 0: Setup

Install `pyyaml` (small new dep for reading the Collector config). LangSmith SDK is already in the repo's pinned deps. Set env vars (same as Labs 17/18).

In [ ]:
# Install dependencies (if not already installed)
# !pip install --quiet pyyaml

import os
import json
import pathlib
import time
import uuid
from datetime import UTC, datetime, timedelta

from dotenv import load_dotenv

# Load .env from repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# LangSmith config — same as Labs 17/18
os.environ["LANGSMITH_TRACING"] = os.environ.get("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "lab-19-online-evaluation")

assert os.environ.get("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY in your .env or shell"

PROJECT = os.environ["LANGSMITH_PROJECT"]
print(f"Project: {PROJECT}")
print(f"Tracing enabled: {os.environ['LANGSMITH_TRACING']}")


## ── Half A: LangSmith online evaluation ──

The SDK polling pattern: fetch recent traces, run a reference-free evaluator, post scores back as feedback. This is the code-side equivalent of a UI-configured Automation Rule.

### Step 1: Generate synthetic traces

In a real deployment, the traces would come from production traffic. For this lab, generate 5 synthetic traces via `client.create_run` so the patterns are reproducible.

Each synthetic trace has an input task, an output answer, and varied citation patterns — some preserve `[1]`/`[2]` markers correctly, some drop them, some fabricate them. This gives us a realistic mix of failure modes for the evaluator to score on.

In [ ]:
from langsmith import Client

client = Client()

# Synthetic production traces with varied citation patterns
synthetic_traces = [
    {
        "name": "research-and-summarize",
        "inputs": {"task": "Research MCP and summarize"},
        "outputs": {
            "answer": "MCP is an open standard [1] with broad production adoption [2]."
        },
        "expected_score": 1.0,  # both markers present
    },
    {
        "name": "research-and-summarize",
        "inputs": {"task": "Research agentic RAG and summarize"},
        "outputs": {
            "answer": "Agentic RAG combines retrieval with agent loops. Multiple frameworks support it."
        },
        "expected_score": 0.0,  # no markers
    },
    {
        "name": "research-and-summarize",
        "inputs": {"task": "Research LangGraph and summarize"},
        "outputs": {
            "answer": "LangGraph is a stateful orchestration runtime [1]."
        },
        "expected_score": 0.5,  # one of two expected markers
    },
    {
        "name": "research-and-summarize",
        "inputs": {"task": "Research observability tools and summarize"},
        "outputs": {
            "answer": "Modern stacks include LangSmith [1], Phoenix [2], Langfuse [3], Laminar [4]."
        },
        "expected_score": 1.0,  # multiple correct markers
    },
    {
        "name": "research-and-summarize",
        "inputs": {"task": "Research OTel and summarize"},
        "outputs": {
            "answer": "OpenTelemetry is a standard [99] with many backends [42]."  # fabricated markers
        },
        "expected_score": 0.5,  # markers present but suspicious
    },
]

# Submit traces via client.create_run
run_ids = []
now = datetime.now(UTC)

for i, t in enumerate(synthetic_traces):
    run_id = uuid.uuid4()
    client.create_run(
        id=run_id,
        name=t["name"],
        run_type="chain",
        inputs=t["inputs"],
        outputs=t["outputs"],
        start_time=now - timedelta(seconds=60 - i * 10),
        end_time=now - timedelta(seconds=50 - i * 10),
        project_name=PROJECT,
    )
    run_ids.append(run_id)
    print(f"  Created run {i+1}: {t['inputs']['task'][:50]}")

print(f"\nCreated {len(run_ids)} traces in project '{PROJECT}'")
print("→ Open smith.langchain.com → check the project; traces appear within ~5s")


### Step 2: A reference-free evaluator

Reuses Lab 16's `citation_preservation` algorithm. Reference-free because it doesn't need a ground-truth answer to compare against — it inspects the output for structural properties (citation markers present and well-formed).

The function signature `(run) -> dict` matches what `client.create_feedback` expects when used in a polling loop.

In [ ]:
import re


def citation_preservation_evaluator(run) -> dict:
    """Reference-free evaluator: checks output for citation marker presence and well-formedness.

    Returns {key, score, comment} where:
    - score = 1.0 if 2+ citation markers present and well-formed
    - score = 0.5 if exactly 1 marker present
    - score = 0.0 if no markers
    - score = 0.3 if markers present but suspiciously high (fabrication signal)
    """
    output = run.outputs or {}
    answer = output.get("answer", "")

    # Find citation-marker-shaped tokens like [1], [2], [42]
    markers = re.findall(r"\[(\d+)\]", answer)
    n_markers = len(markers)
    max_marker = max((int(m) for m in markers), default=0)

    # Heuristic: markers numbered higher than 10 suggest fabrication
    # (real citations are usually small numbers in a sequence)
    if max_marker > 10:
        return {
            "key": "citation_preservation",
            "score": 0.3,
            "comment": f"{n_markers} markers found; max={max_marker} suggests fabrication",
        }
    if n_markers >= 2:
        return {"key": "citation_preservation", "score": 1.0, "comment": f"{n_markers} markers present"}
    if n_markers == 1:
        return {"key": "citation_preservation", "score": 0.5, "comment": "Only 1 marker present"}
    return {"key": "citation_preservation", "score": 0.0, "comment": "No citation markers found"}


# Test on the synthetic outputs directly (before fetching from LangSmith)
print("Evaluator output on synthetic data:")
for t in synthetic_traces:
    # Mock a run object with the same attribute shape
    class MockRun:
        outputs = t["outputs"]
    result = citation_preservation_evaluator(MockRun)
    print(f"  '{t['inputs']['task'][:40]:40}' → score={result['score']:.1f}  ({result['comment']})")


### Step 3: The SDK polling pattern

The canonical "online evaluator" code pattern:

1. `client.list_runs(project_name=...)` — fetch recent traces. `execution_order=1` restricts to root runs (not nested LLM call sub-runs).
2. Iterate, run the evaluator per trace.
3. `client.create_feedback(run_id, key=..., score=...)` — post the score back to LangSmith as feedback on the trace.

This is what a LangSmith Automation Rule does on the platform side. In code, you control the polling cadence, the batching, and the error handling explicitly.

In [ ]:
# Wait a moment to ensure the synthetic traces have been ingested
time.sleep(3)

# Pull recent root runs from the project
recent_runs = list(client.list_runs(
    project_name=PROJECT,
    execution_order=1,  # root runs only
    limit=10,
))

print(f"Found {len(recent_runs)} recent runs in '{PROJECT}'")

# Score each one and post feedback back to LangSmith
for run in recent_runs:
    result = citation_preservation_evaluator(run)
    client.create_feedback(
        run.id,
        key=result["key"],
        score=result["score"],
        comment=result["comment"],
    )
    task = (run.inputs or {}).get("task", "(no task)")
    print(f"  {task[:50]:50} → score={result['score']:.1f}  feedback posted")

print("\n→ In LangSmith UI: each trace now has a citation_preservation feedback score visible.")


**Sample output**:

```
Found 5 recent runs in 'lab-19-online-evaluation'
  Research observability tools and summarize       → score=1.0  feedback posted
  Research OTel and summarize                      → score=0.3  feedback posted
  Research LangGraph and summarize                 → score=0.5  feedback posted
  Research agentic RAG and summarize               → score=0.0  feedback posted
  Research MCP and summarize                       → score=1.0  feedback posted

→ In LangSmith UI: each trace now has a citation_preservation feedback score visible.
```

In the LangSmith UI, the trace detail panel now shows a feedback section with `citation_preservation: 0.5` (or whichever score). Identical to what a UI-configured Rule would produce. You can filter the project by `feedback.citation_preservation < 0.5` to find the problematic traces.

### Step 4: Audit the feedback

After backfilling — or after any online evaluator run — you usually want to verify the scores attached correctly. Pull the runs back with feedback included and inspect.

In [ ]:
# Refetch runs to see attached feedback
audit_runs = list(client.list_runs(
    project_name=PROJECT,
    execution_order=1,
    limit=10,
))

print("Audit of feedback attachment:")
print(f"{'Task':<45} {'Score':<8} {'Comment'}")
print("─" * 90)
for run in audit_runs:
    feedback_list = list(client.list_feedback(run_ids=[run.id]))
    citation_fb = next((f for f in feedback_list if f.key == "citation_preservation"), None)
    task = (run.inputs or {}).get("task", "(no task)")[:43]
    if citation_fb:
        print(f"{task:<45} {citation_fb.score:<8.1f} {citation_fb.comment}")
    else:
        print(f"{task:<45} {'(none)':<8} (no citation_preservation feedback)")


### Step 5: The equivalent LangSmith Automation Rule (UI walkthrough)

The code above is the SDK polling form. The equivalent UI Rule looks like this:

> **Project**: `lab-19-online-evaluation`
> **Tab**: Tracing → Automations → New rule
>
> **Filter**: `(no filter — applies to all traces)`
> **Sample rate**: `100%`
> **Action**: Run custom code evaluator
>   - **Code**: a Python function with the same body as `citation_preservation_evaluator` above
>   - **Upload via**: paste into the LangSmith Rule editor

LangSmith polls the project for new runs, applies the filter, samples at the configured rate, runs the evaluator, posts feedback. Same outcome as the SDK code — different operational model.

**When to use UI Rules**:
- You don't want to maintain custom infrastructure.
- The evaluator logic fits the built-in types (LLM-as-judge with a prompt, or a small Python function).
- Sample rate and filter are simple.

**When to use the SDK polling**:
- You're backfilling historical traces (Rules fire forward only).
- The evaluator depends on external state (databases, APIs) that doesn't fit the Rule editor.
- You already have a Python worker / cron job in your operational model.
- You want explicit batching, retries, ordering guarantees.

Most production teams use both — UI Rules for the standard online-evaluation patterns, SDK code for custom logic and backfills.

### Step 6: Sample-rate decisions and cost arithmetic

LLM-as-judge evaluators cost ~$0.005 per evaluation at gpt-4o-mini rates. At production scale, the sample rate is a real cost lever. Compute the math for a representative deployment:

In [ ]:
# Cost model for an LLM-as-judge online evaluator at varying sample rates
# Assumptions: 1M traces/month, $0.005 per LLM-as-judge evaluation
traces_per_month = 1_000_000
cost_per_eval_usd = 0.005

scenarios = [
    ("Evaluate 100% of traces", 1.00),
    ("Evaluate 10% of traces (typical)", 0.10),
    ("Evaluate 1% baseline", 0.01),
    ("Evaluate 100% of errors only (assume 0.5% error rate)", 0.005),
]

print(f"{'Scenario':<55} {'Monthly cost':<18} {'Notes'}")
print("─" * 100)
for name, rate in scenarios:
    evals_per_month = traces_per_month * rate
    monthly_cost = evals_per_month * cost_per_eval_usd
    print(f"{name:<55} ${monthly_cost:>10,.0f}     {evals_per_month:,.0f} evals/mo")


**Sample output**:

```
Scenario                                                Monthly cost       Notes
────────────────────────────────────────────────────────────────────────────────────────────────────
Evaluate 100% of traces                                 $     5,000     1,000,000 evals/mo
Evaluate 10% of traces (typical)                        $       500     100,000 evals/mo
Evaluate 1% baseline                                    $        50     10,000 evals/mo
Evaluate 100% of errors only (assume 0.5% error rate)   $        25     5,000 evals/mo
```

**The pattern**: an aggressive sample rate is often the right starting point. 1% gives you enough volume for dashboard aggregates and trend detection. 100% on errors only catches every failure with full fidelity for human review. Together: $75/mo for diagnostic coverage that 100% sampling would charge $5,000 for.

Rule-based (free) evaluators like `citation_preservation` don't need this calculation — running them at 100% is cheap. The cost model is specifically for LLM-as-judge evaluators where the per-evaluation cost is non-trivial.

## ── Half B: Tail-based sampling at the OTel Collector ──

Online evaluators (Half A) decide WHAT to do with traces after the platform has them. Tail-based sampling decides which traces reach the platform in the first place. Different layer; complementary pattern.

### Step 7: The data flow

```mermaid
flowchart LR
    A[Application<br/>emits 100% of spans] --> B[OTel Collector<br/>tail_sampling processor]
    B -->|kept ~7%| C[LangSmith]
    B -->|kept ~7%| D[Datadog]
    B -->|kept ~7%| E[Langfuse]
    B -.->|dropped ~93%| F[discarded]
```

The Collector sits between the application and the backends. The application has no idea whether a trace will be kept — it emits everything. The Collector waits for the trace to complete (`decision_wait`), evaluates policies, decides.

The same decision applies uniformly to every downstream backend. Fanout to LangSmith + Datadog + Langfuse all see the same sampled subset.

### Step 8: A real otel-collector-config.yaml

This is the YAML config that drives the tail sampling. Read each policy; note the order (first-match-wins); note the `decision_wait` and `num_traces` memory-budget knobs.

In [ ]:
import yaml

otel_collector_config = """
receivers:
  otlp:
    protocols:
      grpc:
        endpoint: 0.0.0.0:4317
      http:
        endpoint: 0.0.0.0:4318

processors:
  tail_sampling:
    # Wait 30s for late spans before deciding. Agent traces with long tool calls
    # need a longer window than the conventional 10s default.
    decision_wait: 30s

    # Memory budget: traces_per_sec * decision_wait * safety_margin (2x)
    # 100/sec * 30s * 2 = 6000 traces in memory
    num_traces: 6000
    expected_new_traces_per_sec: 100

    # Policies evaluated in order; FIRST MATCH WINS
    policies:
      # ──── KEEP 100% of these ────

      - name: errors
        type: status_code
        status_code:
          status_codes: [ERROR]

      - name: high-latency
        type: latency
        latency:
          threshold_ms: 30000

      - name: high-token-usage
        type: numeric_attribute
        numeric_attribute:
          key: gen_ai.usage.input_tokens
          min_value: 5000

      - name: debug-users
        type: string_attribute
        string_attribute:
          key: user.id
          values: [debug-1, debug-2, beta-cohort-a]

      # ──── BASELINE: 5% of everything else ────

      - name: baseline-sample
        type: probabilistic
        probabilistic:
          sampling_percentage: 5

exporters:
  otlp/langsmith:
    endpoint: https://api.smith.langchain.com/otel
    headers:
      x-api-key: ${LANGSMITH_API_KEY}
      Langsmith-Project: lab-19

service:
  pipelines:
    traces:
      receivers: [otlp]
      processors: [tail_sampling]
      exporters: [otlp/langsmith]
"""

config = yaml.safe_load(otel_collector_config)
policies = config["processors"]["tail_sampling"]["policies"]

print(f"tail_sampling configured with {len(policies)} policies:")
for i, p in enumerate(policies, 1):
    print(f"  {i}. {p['name']:25} (type: {p['type']})")

print(f"\ndecision_wait: {config['processors']['tail_sampling']['decision_wait']}")
print(f"num_traces budget: {config['processors']['tail_sampling']['num_traces']:,}")


### Step 9: Simulate the policy logic in Python

The tail-sampling processor evaluates each completed trace against the policies in order. First match wins. Implementing this in Python lets us see exactly what would get kept vs dropped, without deploying a real Collector.

In [ ]:
import random

random.seed(42)  # deterministic output


def simulate_tail_sampler(traces: list, policies: list) -> dict:
    """Apply tail-sampling policy logic to a list of synthetic trace summaries.

    Each trace is a dict with: trace_id, status, latency_ms, input_tokens, user_id.
    Returns {kept: [...], dropped: [...], matched_by: {policy_name: count}}.
    """
    kept = []
    dropped = []
    matched_by = {p["name"]: 0 for p in policies}

    for trace in traces:
        # Try each policy in order; first match wins
        kept_by = None
        for policy in policies:
            ptype = policy["type"]

            if ptype == "status_code":
                wanted = policy["status_code"]["status_codes"]
                if trace["status"] in wanted:
                    kept_by = policy["name"]
                    break

            elif ptype == "latency":
                threshold = policy["latency"]["threshold_ms"]
                if trace["latency_ms"] >= threshold:
                    kept_by = policy["name"]
                    break

            elif ptype == "numeric_attribute":
                key = policy["numeric_attribute"]["key"]
                min_value = policy["numeric_attribute"]["min_value"]
                if trace.get(key.replace("gen_ai.usage.", ""), 0) >= min_value:
                    kept_by = policy["name"]
                    break

            elif ptype == "string_attribute":
                key = policy["string_attribute"]["key"]
                values = policy["string_attribute"]["values"]
                if trace.get(key.replace("user.", "user_"), None) in values:
                    kept_by = policy["name"]
                    break

            elif ptype == "probabilistic":
                rate = policy["probabilistic"]["sampling_percentage"] / 100
                if random.random() < rate:
                    kept_by = policy["name"]
                    break

        if kept_by:
            matched_by[kept_by] += 1
            kept.append((trace, kept_by))
        else:
            dropped.append(trace)

    return {"kept": kept, "dropped": dropped, "matched_by": matched_by}


# Generate 1000 synthetic trace summaries representing a slice of production traffic
def generate_synthetic_traces(n=1000):
    traces = []
    for i in range(n):
        # Realistic distribution: 0.5% errors, 0.3% high-latency, 1% high-token-usage,
        # 0.1% debug users; rest are happy-path
        r = random.random()
        if r < 0.005:
            status = "ERROR"
            latency = random.randint(500, 30000)
            input_tokens = random.randint(200, 3000)
        elif r < 0.008:
            status = "OK"
            latency = random.randint(30000, 60000)
            input_tokens = random.randint(200, 3000)
        elif r < 0.018:
            status = "OK"
            latency = random.randint(500, 5000)
            input_tokens = random.randint(5000, 12000)
        else:
            status = "OK"
            latency = random.randint(500, 5000)
            input_tokens = random.randint(200, 3000)
        user_id = random.choice(["debug-1", "debug-2"]) if random.random() < 0.001 else f"user_{i}"
        traces.append({
            "trace_id": f"trace_{i:04d}",
            "status": status, "latency_ms": latency,
            "input_tokens": input_tokens, "user_id": user_id,
        })
    return traces


synthetic = generate_synthetic_traces(1000)
result = simulate_tail_sampler(synthetic, policies)

print(f"Total traces:    {len(synthetic):,}")
print(f"  Kept:          {len(result['kept']):>5,}  ({100*len(result['kept'])/len(synthetic):.1f}%)")
print(f"  Dropped:       {len(result['dropped']):>5,}  ({100*len(result['dropped'])/len(synthetic):.1f}%)")
print("\nBreakdown of kept traces by policy:")
for name, count in result["matched_by"].items():
    print(f"  {name:25} {count:>5,} traces")


**Sample output**:

```
Total traces:    1,000
  Kept:            71  (7.1%)
  Dropped:        929  (92.9%)

Breakdown of kept traces by policy:
  errors                        4 traces
  high-latency                  3 traces
  high-token-usage             10 traces
  debug-users                   1 traces
  baseline-sample              53 traces
```

The policy stack does its job:
- **All 4 errors** are kept (100% retention on the diagnostic gold).
- **All 3 high-latency traces** are kept.
- **All 10 high-token-usage traces** are kept.
- **All 1 debug-user traces** kept.
- **53 of the remaining ~982 happy-path traces** kept via 5% baseline.

Net: 7.1% retained, 92.9% dropped. At 1M traces/mo and $0.001/trace storage, that's $930/mo saved — and 100% of errors still in the dataset for human review.

### Step 10: The load-balancing constraint

The constraint that bites teams at scale: **all spans for a given trace must reach the same Collector instance.** Tail sampling needs to see the whole trace to evaluate latency (start-to-end span), to look at every span's status, to check whether any span has a high-token-usage attribute. If half a trace's spans go to Collector A and the other half to Collector B, neither has enough info to decide correctly.

```
WRONG: Application → Load Balancer (round-robin) → [Collector A | Collector B | Collector C]
         Each Collector sees a fraction of each trace.
         Tail sampling decisions are based on incomplete data.
         Some error traces get partially dropped.
         Failure mode: "tail sampling looks broken but config is correct".

RIGHT: Application → Agent Collector → loadbalancingexporter (route by trace_id)
                                                    ↓
                              [Tail-sampling Collector A | B | C] → Backends
         Each Collector sees ALL spans for the traces it owns.
         Tail sampling decisions are based on complete traces.
```

The fix is the two-tier topology with the `loadbalancingexporter`. The first-tier (agent) Collectors are lightweight; they just receive spans and forward them to the right second-tier (tail-sampling) Collector based on trace_id hash. The second tier has the full traces in its buffer and decides correctly.

In single-instance deployments — i.e., one Collector receives all spans — this isn't a concern. The problem only appears when you scale horizontally for throughput. Surprisingly common in production deployments that scaled before reading the manual.

(Not implemented as code in this lab — running a multi-instance Collector setup requires docker + multiple nodes. The architectural understanding is what matters.)

## Step 11: Synthesis

What this lab built:

**Half A — LangSmith online evaluation**:
- Synthetic trace generation via `client.create_run` (production-traffic stand-in for course labs).
- The SDK polling pattern: `list_runs` + iterate + `create_feedback` — code-side equivalent of an Automation Rule.
- A reference-free evaluator (`citation_preservation`) reused from Lab 16.
- The UI Rule walkthrough showing what the equivalent click-configuration looks like.
- Cost arithmetic for sample-rate decisions on LLM-as-judge evaluators.

**Half B — OTel Collector tail sampling**:
- A real 5-policy `tail_sampling` YAML config.
- A Python simulator implementing the same first-match-wins policy logic against synthetic trace summaries.
- The kept-vs-dropped breakdown: errors / high-latency / high-token-usage / debug-users at 100%; 5% baseline for everything else; net ~7% retention.
- The load-balancing constraint and the two-tier topology that solves it.

**When each pattern earns its place**:

- **LangSmith Rules** are right when: the storage and ingestion cost is acceptable (small to medium scale); you want platform-side workflows (annotation queue routing, dataset promotion, webhook fanout to alerting); the evaluator logic fits the platform's built-in types or a small custom code function.
- **Collector tail sampling** is right when: storage and ingestion cost is the binding constraint (high-volume production); you need vendor-agnostic sampling that applies to every backend; compliance requires retaining 100% of errors but not happy-path traces.
- **Both together** is the production pattern at scale: tail-sample at the Collector to control storage; register Rules at the platform to drive workflows. They're complementary, not competing.

What this lab didn't cover (deferred):

- **Actual Collector deployment.** Docker + multi-node setup; out of scope for a notebook.
- **LangSmith Engine** — the AI layer on top of online evaluators. Concept page mentions it; deeper coverage in future content.
- **Drift detection on the sampled subset.** Module 5.
- **Agent-as-judge calibration against periodic human labels.** Module 5.
- **OTel baggage for cost attribution.** Module 6.

Path 06 Module 4 is now complete. Module 5 (planned, future batch) takes the output of these online evaluators and asks: how do you detect when the scores are drifting, and how do you calibrate LLM-as-judge against human ground truth?

✓ **Module 4 complete.**
